# DA6401 — W&B Report: Novel Images (2.7) + Meta-Analysis (2.8)

**BEFORE RUNNING:**
1. Settings -> Accelerator -> **GPU T4 x2**
2. Settings -> Internet -> **ON**
3. Click **Save Version -> Save & Run All (Commit)**

In [ ]:
import os
os.environ["WANDB_API_KEY"]      = "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"
os.environ["WANDB_DISABLE_CODE"] = "1"

CKPT = "/kaggle/working/checkpoints"
os.makedirs(CKPT, exist_ok=True)

!git clone https://github.com/usnaveen/A2_Deep_Learning.git /kaggle/working/repo 2>&1 | tail -3
%cd /kaggle/working/repo
!pip install -q wandb albumentations gdown scikit-learn

import torch, numpy as np
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
import gdown, re

CLASSIFIER_ID = "1x_qgIKPFwu_2inuIjyms7IhQY2Qr_ZKP"
LOCALIZER_ID  = "1jRM83Xr6HETheUhv1MQqRMEl2eAwbkSK"
UNET_ID       = "1ATvVvuMPagLhP3wLjI-sWe3D1Z4wHV0Z"

for did, name in [(CLASSIFIER_ID,'classifier.pth'),(LOCALIZER_ID,'localizer.pth'),(UNET_ID,'unet.pth')]:
    out = f"{CKPT}/{name}"
    gdown.download(id=did, output=out, quiet=True, fuzzy=True)
    print(f"  {name}: {os.path.getsize(out)/1e6:.0f} MB")

In [ ]:
def _canonicalize_checkpoint(sd):
    """Normalize nested-block key format to our flat convention."""
    if not any(re.match(r'encoder\.block\d+\.\d+\.\d+\.', k) for k in sd):
        return sd
    new_sd = {}
    for k, v in sd.items():
        m = re.match(r'encoder\.block(\d+)\.(\d+)\.(\d+)\.(.*)', k)
        if m:
            N, M, L, suf = int(m.group(1)), int(m.group(2)), int(m.group(3)), m.group(4)
            X = N - 1
            if X in (0, 1):
                Y = 0 if L == 0 else 1
            else:
                Y = (0 if L == 0 else 1) if M == 0 else (3 if L == 0 else 4)
            new_sd[f'encoder.block{X}.{Y}.{suf}'] = v
        elif k.startswith('classifier.'):
            new_sd['head.' + k[len('classifier.'):]] = v
        elif k.startswith('regressor.'):
            new_sd['head.' + k[len('regressor.'):]] = v
    return new_sd

# Load models
from models.classification import VGG11Classifier
from models.localization import VGG11Localizer
from models.segmentation import VGG11UNet

cls_model = VGG11Classifier(num_classes=37, dropout_p=0.5, use_bn=True).to(device)
loc_model = VGG11Localizer(image_size=224).to(device)
seg_model = VGG11UNet(num_classes=3).to(device)

cls_sd = _canonicalize_checkpoint(torch.load(f'{CKPT}/classifier.pth', map_location=device, weights_only=False))
cls_model.load_state_dict(cls_sd, strict=False)

loc_sd = _canonicalize_checkpoint(torch.load(f'{CKPT}/localizer.pth', map_location=device, weights_only=False))
loc_model.load_state_dict(loc_sd, strict=False)

seg_model.load_state_dict(torch.load(f'{CKPT}/unet.pth', map_location=device, weights_only=False), strict=False)

cls_model.eval(); loc_model.eval(); seg_model.eval()
print('All 3 models loaded.')

---
## 2.7 — Novel Image Pipeline Showcase (5 marks)

3 novel pet images downloaded from the internet (NOT from the dataset).
Each image goes through: Classification -> Localization -> Segmentation.

In [ ]:
import urllib.request
from PIL import Image as PILImage

NOVEL_DIR = '/kaggle/working/novel_images'
os.makedirs(NOVEL_DIR, exist_ok=True)

# 3 novel pet images from the internet (not Oxford Pet dataset)
NOVEL_URLS = [
    ('golden_retriever.jpg',
     'https://upload.wikimedia.org/wikipedia/commons/thumb/b/bd/Golden_Retriever_Dukedestiny01_dbread_08.jpg/800px-Golden_Retriever_Dukedestiny01_dread_08.jpg'),
    ('tabby_cat.jpg',
     'https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/800px-Cat_November_2010-1a.jpg'),
    ('pug_dog.jpg',
     'https://upload.wikimedia.org/wikipedia/commons/thumb/f/f0/Mops_oct09_cropped2.jpg/800px-Mops_oct09_cropped2.jpg'),
]

novel_paths = []
for fname, url in NOVEL_URLS:
    out_path = f'{NOVEL_DIR}/{fname}'
    try:
        urllib.request.urlretrieve(url, out_path)
        img = PILImage.open(out_path)
        print(f'  {fname}: {img.size[0]}x{img.size[1]}')
        novel_paths.append(out_path)
    except Exception as e:
        print(f'  {fname}: FAILED - {e}')

# Fallback: if any URL fails, create simple test images from val set
if len(novel_paths) < 3:
    print('\nSome downloads failed. Using fallback from web search...')
    FALLBACK_URLS = [
        ('pet1.jpg', 'https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Cat03.jpg/1200px-Cat03.jpg'),
        ('pet2.jpg', 'https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/YellowLabradorLooking_new.jpg/1200px-YellowLabradorLooking_new.jpg'),
        ('pet3.jpg', 'https://upload.wikimedia.org/wikipedia/commons/thumb/1/18/Dog_park_in_San_Jose.jpg/1200px-Dog_park_in_San_Jose.jpg'),
    ]
    for fname, url in FALLBACK_URLS:
        if len(novel_paths) >= 3:
            break
        out_path = f'{NOVEL_DIR}/{fname}'
        try:
            urllib.request.urlretrieve(url, out_path)
            novel_paths.append(out_path)
            print(f'  Fallback {fname}: OK')
        except:
            pass

print(f'\n{len(novel_paths)} novel images ready.')

In [ ]:
import wandb
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from io import BytesIO
from data.pets_dataset import OxfordIIITPetDataset, IMAGENET_MEAN, IMAGENET_STD
from train import denormalize_image

IMAGE_SIZE = 224
class_names = OxfordIIITPetDataset.CLASS_NAMES

# Trimap colormap: 0=foreground(green), 1=background(dark), 2=boundary(yellow)
colormap = np.array([[0, 200, 0], [40, 40, 40], [255, 255, 0]], dtype=np.uint8)

transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

wandb.init(project='da6401-assignment2',
           name='section_2.7_novel_images',
           tags=['exp-novel', 'report'])

result_images = []

for img_path in novel_paths:
    fname = os.path.basename(img_path)
    orig_img = np.array(PILImage.open(img_path).convert('RGB'))
    transformed = transform(image=orig_img)
    img_tensor = transformed['image'].unsqueeze(0).to(device)

    with torch.no_grad():
        # Classification
        cls_logits = cls_model(img_tensor)
        cls_probs = torch.softmax(cls_logits, dim=1)
        conf, pred_idx = cls_probs.max(dim=1)
        breed = class_names[pred_idx.item()]
        confidence = conf.item()

        # Top-3 predictions
        top3_vals, top3_idx = cls_probs.topk(3, dim=1)
        top3_str = ', '.join([f'{class_names[i]}({v:.2f})' for v, i in
                              zip(top3_vals[0].tolist(), top3_idx[0].tolist())])

        # Localization
        bbox = loc_model(img_tensor).cpu().numpy()[0]  # [cx, cy, w, h]

        # Segmentation
        seg_mask = seg_model(img_tensor).argmax(dim=1).cpu().numpy()[0]

    # ---- Visualization: 4-panel figure ----
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    # Panel 1: Original image
    resized = np.array(PILImage.fromarray(orig_img).resize((IMAGE_SIZE, IMAGE_SIZE)))
    axes[0].imshow(resized)
    axes[0].set_title('Original', fontsize=12)
    axes[0].axis('off')

    # Panel 2: Classification result
    axes[1].imshow(resized)
    axes[1].set_title(f'Pred: {breed}\nConf: {confidence:.3f}', fontsize=11,
                      color='green' if confidence > 0.5 else 'red')
    axes[1].axis('off')

    # Panel 3: Bounding box
    img_denorm = denormalize_image(img_tensor[0])
    axes[2].imshow(img_denorm)
    x1 = bbox[0] - bbox[2] / 2
    y1 = bbox[1] - bbox[3] / 2
    rect = patches.Rectangle((x1, y1), bbox[2], bbox[3],
                              linewidth=3, edgecolor='red', facecolor='none')
    axes[2].add_patch(rect)
    axes[2].set_title('Bounding Box', fontsize=12)
    axes[2].axis('off')

    # Panel 4: Segmentation mask overlay
    mask_colored = colormap[seg_mask]
    overlay = (0.5 * resized + 0.5 * mask_colored).astype(np.uint8)
    axes[3].imshow(overlay)
    axes[3].set_title('Segmentation Overlay', fontsize=12)
    axes[3].axis('off')

    plt.suptitle(f'{fname} — Top-3: {top3_str}', fontsize=13, fontweight='bold')
    plt.tight_layout()

    buf = BytesIO()
    fig.savefig(buf, format='png', dpi=150, bbox_inches='tight')
    buf.seek(0)
    plt.close(fig)

    result_images.append(wandb.Image(
        PILImage.open(buf),
        caption=f'{fname}: {breed} ({confidence:.2f})'))

    print(f'  {fname}: {breed} ({confidence:.3f}) | bbox=[{bbox[0]:.0f},{bbox[1]:.0f},{bbox[2]:.0f},{bbox[3]:.0f}]')

wandb.log({'novel_images/pipeline_output': result_images})
print('\nNovel images logged to W&B.')

---
## 2.8 — Meta-Analysis: Training vs Validation Plots (10 marks)

Pull metrics from all W&B training runs, create overlaid comparison plots.

In [ ]:
import wandb
api = wandb.Api()

PROJECT = 'da25m020-indian-institute-of-technology-madras/da6401-assignment2'

# Pull all runs
runs = api.runs(PROJECT)
print(f'Found {len(runs)} runs in project.\n')

for r in runs:
    tags = ', '.join(r.tags) if r.tags else 'none'
    print(f'  {r.name:45s} | state={r.state:10s} | tags={tags}')

In [ ]:
import pandas as pd

# Helper to pull run history
def get_run_history(run_name_or_id, keys):
    """Pull metric history from a W&B run."""
    for r in runs:
        if r.name == run_name_or_id or r.id == run_name_or_id:
            hist = r.history(keys=keys, pandas=True)
            return hist
    return None

def find_runs_with_tag(tag):
    """Find all runs with a specific tag."""
    return [r for r in runs if tag in (r.tags or [])]

# ---- Classification: Train vs Val Loss, F1 ----
cls_runs = find_runs_with_tag('classification')
if not cls_runs:
    cls_runs = [r for r in runs if 'classif' in r.name.lower() or 'dropout' in r.name.lower()]

print(f'Classification runs: {[r.name for r in cls_runs]}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for r in cls_runs[:5]:  # limit to 5 runs
    h = r.history(pandas=True)
    if h.empty:
        continue
    # Loss
    if 'train/loss' in h.columns and 'val/loss' in h.columns:
        axes[0].plot(h['train/loss'].dropna(), label=f'{r.name} (train)', linestyle='--', alpha=0.7)
        axes[0].plot(h['val/loss'].dropna(), label=f'{r.name} (val)', alpha=0.9)
    # F1 / Accuracy
    for col in ['val/f1', 'val/accuracy', 'val/acc']:
        if col in h.columns:
            axes[1].plot(h[col].dropna(), label=f'{r.name}', alpha=0.9)
            break

axes[0].set_title('Classification: Train vs Val Loss', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

axes[1].set_title('Classification: Validation Metric', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Score')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
buf = BytesIO(); fig.savefig(buf, format='png', dpi=150); buf.seek(0); plt.close(fig)
wandb.log({'meta_analysis/classification_curves': wandb.Image(PILImage.open(buf))})
print('Classification plots logged.')

In [ ]:
# ---- Localization: Train vs Val Loss, IoU ----
loc_runs = find_runs_with_tag('localization')
if not loc_runs:
    loc_runs = [r for r in runs if 'local' in r.name.lower() or 'detect' in r.name.lower()]

print(f'Localization runs: {[r.name for r in loc_runs]}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for r in loc_runs[:5]:
    h = r.history(pandas=True)
    if h.empty:
        continue
    if 'train/loss' in h.columns and 'val/loss' in h.columns:
        axes[0].plot(h['train/loss'].dropna(), label=f'{r.name} (train)', linestyle='--', alpha=0.7)
        axes[0].plot(h['val/loss'].dropna(), label=f'{r.name} (val)', alpha=0.9)
    for col in ['val/iou', 'val/mean_iou']:
        if col in h.columns:
            axes[1].plot(h[col].dropna(), label=f'{r.name}', alpha=0.9)
            break

axes[0].set_title('Localization: Train vs Val Loss', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

axes[1].set_title('Localization: Validation IoU', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('IoU')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
buf = BytesIO(); fig.savefig(buf, format='png', dpi=150); buf.seek(0); plt.close(fig)
wandb.log({'meta_analysis/localization_curves': wandb.Image(PILImage.open(buf))})
print('Localization plots logged.')

In [ ]:
# ---- Segmentation: Train vs Val Loss, Dice ----
seg_runs = find_runs_with_tag('segmentation')
if not seg_runs:
    seg_runs = [r for r in runs if 'seg' in r.name.lower() or 'unet' in r.name.lower()]

print(f'Segmentation runs: {[r.name for r in seg_runs]}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for r in seg_runs[:5]:
    h = r.history(pandas=True)
    if h.empty:
        continue
    if 'train/loss' in h.columns and 'val/loss' in h.columns:
        axes[0].plot(h['train/loss'].dropna(), label=f'{r.name} (train)', linestyle='--', alpha=0.7)
        axes[0].plot(h['val/loss'].dropna(), label=f'{r.name} (val)', alpha=0.9)
    for col in ['val/dice', 'val/mean_dice']:
        if col in h.columns:
            axes[1].plot(h[col].dropna(), label=f'{r.name}', alpha=0.9)
            break

axes[0].set_title('Segmentation: Train vs Val Loss', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

axes[1].set_title('Segmentation: Validation Dice', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Dice Score')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
buf = BytesIO(); fig.savefig(buf, format='png', dpi=150); buf.seek(0); plt.close(fig)
wandb.log({'meta_analysis/segmentation_curves': wandb.Image(PILImage.open(buf))})
print('Segmentation plots logged.')

In [ ]:
# ---- Summary comparison: best metrics per task across all systems ----
fig, ax = plt.subplots(figsize=(10, 6))

# Collect best metrics from each run
summary_data = []
for r in runs:
    s = r.summary
    entry = {'name': r.name, 'tags': r.tags or []}
    for key in ['val/f1', 'val/accuracy', 'val/acc', 'val/iou', 'val/mean_iou',
                'val/dice', 'val/mean_dice', 'best_val_f1', 'best_val_iou', 'best_val_dice']:
        if key in s:
            entry[key] = s[key]
    summary_data.append(entry)

# Print summary table
print(f'{"Run":45s} | {"F1":>6s} | {"IoU":>6s} | {"Dice":>6s}')
print('-' * 75)
for e in summary_data:
    f1  = e.get('val/f1', e.get('val/accuracy', e.get('best_val_f1', '')))
    iou = e.get('val/iou', e.get('val/mean_iou', e.get('best_val_iou', '')))
    dice = e.get('val/dice', e.get('val/mean_dice', e.get('best_val_dice', '')))
    f1_s = f'{f1:.4f}' if isinstance(f1, (int, float)) else '-'
    iou_s = f'{iou:.4f}' if isinstance(iou, (int, float)) else '-'
    dice_s = f'{dice:.4f}' if isinstance(dice, (int, float)) else '-'
    print(f'{e["name"]:45s} | {f1_s:>6s} | {iou_s:>6s} | {dice_s:>6s}')

# Create a grouped bar chart of final metrics
tasks = ['Classification (F1)', 'Localization (IoU)', 'Segmentation (Dice)']
thresholds = [0.80, 0.50, 0.30]
best_scores = [0, 0, 0]

for e in summary_data:
    f1  = e.get('val/f1', e.get('best_val_f1', 0)) or 0
    iou = e.get('val/iou', e.get('best_val_iou', 0)) or 0
    dice = e.get('val/dice', e.get('best_val_dice', 0)) or 0
    if isinstance(f1, (int, float)): best_scores[0] = max(best_scores[0], f1)
    if isinstance(iou, (int, float)): best_scores[1] = max(best_scores[1], iou)
    if isinstance(dice, (int, float)): best_scores[2] = max(best_scores[2], dice)

x = np.arange(len(tasks))
bars = ax.bar(x, best_scores, width=0.5, color=['#2196F3', '#4CAF50', '#FF9800'], alpha=0.85)
for i, (thr, score) in enumerate(zip(thresholds, best_scores)):
    ax.axhline(y=thr, xmin=i/3+0.05, xmax=(i+1)/3-0.05,
               color='red', linestyle='--', linewidth=1.5, alpha=0.7)
    ax.text(i, score + 0.02, f'{score:.3f}', ha='center', fontsize=12, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(tasks, fontsize=11)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Best Validation Scores Across All Training Runs', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.legend(['Threshold'], fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
buf = BytesIO(); fig.savefig(buf, format='png', dpi=150); buf.seek(0); plt.close(fig)
wandb.log({'meta_analysis/best_scores_summary': wandb.Image(PILImage.open(buf))})
print('\nSummary chart logged.')

In [ ]:
import torch.nn as nn

# ---- Feature Map Visualization (for 2.4 / 2.8) ----
# Show first conv and last conv feature maps
activations = {}
def make_hook(name):
    def hook_fn(module, inp, out):
        activations[name] = out.detach().cpu()
    return hook_fn

cls_model.encoder.block0[0].register_forward_hook(make_hook('first_conv'))
last_idx = max(i for i, m in enumerate(cls_model.encoder.block4) if isinstance(m, nn.Conv2d))
cls_model.encoder.block4[last_idx].register_forward_hook(make_hook('last_conv'))

# Use first novel image
test_img = transform(image=np.array(PILImage.open(novel_paths[0]).convert('RGB')))['image']
test_img = test_img.unsqueeze(0).to(device)
with torch.no_grad():
    _ = cls_model(test_img)

for layer_name in ['first_conv', 'last_conv']:
    feat = activations[layer_name][0]  # [C, H, W]
    n_ch = min(32, feat.shape[0])
    n_cols = 8
    n_rows = (n_ch + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*2, n_rows*2))
    axes = axes.flatten()
    for c in range(n_ch):
        fm = feat[c].numpy()
        fm = (fm - fm.min()) / (fm.max() - fm.min() + 1e-8)
        axes[c].imshow(fm, cmap='viridis')
        axes[c].set_title(f'Ch {c}', fontsize=7)
        axes[c].axis('off')
    for c in range(n_ch, len(axes)):
        axes[c].axis('off')
    plt.suptitle(f'Feature Maps: {layer_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    buf = BytesIO(); fig.savefig(buf, format='png', dpi=150); buf.seek(0); plt.close(fig)
    wandb.log({f'meta_analysis/feature_maps_{layer_name}': wandb.Image(PILImage.open(buf))})

print('Feature maps logged.')

In [ ]:
wandb.finish()
print('='*55)
print('  All W&B report sections logged!')
print('='*55)
print()
print('  Section 2.7: novel_images/pipeline_output')
print('  Section 2.8: meta_analysis/classification_curves')
print('               meta_analysis/localization_curves')
print('               meta_analysis/segmentation_curves')
print('               meta_analysis/best_scores_summary')
print('               meta_analysis/feature_maps_first_conv')
print('               meta_analysis/feature_maps_last_conv')
print()
print('  Go to W&B -> create report -> drag these panels in.')